# LLM-from-Scratch (124M Parameters)

Train a GPT-style Transformer from scratch on Google Colab using FineWeb-Edu dataset.

**Features:**
- **~124M parameters** decoder-only Transformer
- **tiktoken (GPT-2)** tokenizer — battle-tested BPE
- **FineWeb-Edu** dataset — high-quality web text
- **Perplexity tracking** + sample generation during training
- **Session-safe** with automatic checkpoint resume
- **torch.compile** for ~1.5× speedup on T4/A100

> **Tip:** Enable GPU (T4/A100) before starting: Runtime → Change runtime type → GPU

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q torch numpy tqdm requests matplotlib regex datasets tiktoken huggingface-hub

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive & Clone Repository

> Drive is used to persist checkpoints and logs across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/llm-from-scratch/checkpoints
!mkdir -p /content/drive/MyDrive/llm-from-scratch/logs

%cd /content
!rm -rf llm-from-scratch
!git clone https://github.com/avneeshjadhav04/llm-from-scratch.git
%cd llm-from-scratch

## 3. Prepare Data (FineWeb-Edu)

> Downloads and tokenizes ~10M tokens from FineWeb-Edu.
> **Takes ~5–10 minutes on first run.** Subsequent runs skip if binaries exist.

In [ ]:
!python prepare_data.py

## 4. Train the 124M Model (Auto-Resume)

> **Smart training block** — automatically handles both first-run and resume:
> 1. Checks Drive for existing checkpoints → copies if found
> 2. Runs `prepare_data.py` (skips if binaries exist)
> 3. Starts/resumes training up to **5,000 steps** per session
> 4. Saves checkpoints and logs back to Drive when done

> Run this same cell every session — fresh or resume.

In [ ]:
import os
import glob
import shutil

DRIVE_CKPT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
DRIVE_LOG_DIR = "/content/drive/MyDrive/llm-from-scratch/logs"

# Ensure local dirs exist
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Check for existing checkpoints in Drive
drive_checkpoints = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/*.pt"))
if drive_checkpoints:
    latest = drive_checkpoints[-1]
    step = int(os.path.basename(latest).split("_")[-1].replace(".pt", ""))
    print(f"Found checkpoint in Drive: {os.path.basename(latest)} (step {step})")
    print("Copying checkpoint and logs from Drive...")
    !cp {DRIVE_CKPT_DIR}/*.pt checkpoints/ 2>/dev/null || true
    !cp {DRIVE_LOG_DIR}/* logs/ 2>/dev/null || true
    print(f"Resuming training from step {step}")
else:
    print("No checkpoint found in Drive. Starting training from scratch.")

# Prepare data (idempotent — skips if binaries exist)
print("\nPreparing dataset...")
!python prepare_data.py

# Train
SESSION_STEPS = 5000
print(f"\nStarting training (up to {SESSION_STEPS} steps this session)...")
!python train.py --max_steps_per_session {SESSION_STEPS}

# Save outputs back to Drive
print("\nSaving checkpoints and logs to Drive...")
for f in os.listdir("checkpoints"):
    if f.endswith(".pt"):
        shutil.copy(f"checkpoints/{f}", DRIVE_CKPT_DIR)
        print(f"  Saved checkpoint: {f}")

for f in os.listdir("logs"):
    shutil.copy(f"logs/{f}", DRIVE_LOG_DIR)
    print(f"  Saved log: {f}")

print("\nDone! Run this same cell in your next session to resume.")

## 5. Generate Text (Interactive)

> Uses the **latest saved checkpoint** automatically.

In [ ]:
!python generate.py \
    --prompt "The future of artificial intelligence is" \
    --temperature 0.8 \
    --top_k 40 \
    --top_p 0.95 \
    --max_new_tokens 256 \
    --device cuda

## 6. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("logs/100m_training_log.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["step"], df["train_loss"], label="Train Loss")
val_mask = df["val_loss"].notna()
axes[0].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_loss"], label="Val Loss", linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df["step"], df["train_ppl"], label="Train PPL")
axes[1].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_ppl"], label="Val PPL", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Training & Validation Perplexity")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()